### 1. Initialisation de Spark et configuration

In [1]:
import os
from decimal import Decimal
from pyspark.sql import SparkSession, functions as F

# Configurer JAVA_HOME
os.environ.setdefault("JAVA_HOME", "/usr/lib/jvm/java-17-openjdk-amd64")
spark = (SparkSession.builder.master("local[2]")
         .appName("TP3-Retail-Intelligence-UC06")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
BASE = "hdfs://localhost:9000/retail-lake"
print("Spark :", spark.version)
print("HDFS :", BASE)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 16:40:14 WARN Utils: Your hostname, DESKTOP-38E0JSC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/18 16:40:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/honi/environnements/retail-tp3/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/18 16:40:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark : 4.3.0
HDFS : hdfs://localhost:9000/retail-lake


### 2. Inventaire des sources et lecture Raw

Les données TPCx-BB sont séparées par `|`, sans en-têtes. `store` a 29 champs, `item` en a 22 et `store_sales` en a 23. La lecture est faite directement depuis HDFS.

In [2]:
sources = {
    "store": (f"{BASE}/raw/nifi/store.dat", 29),
    "item": (f"{BASE}/raw/tpcxbb/initial/item/item.dat", 22),
    "store_sales": (f"{BASE}/raw/tpcxbb/initial/store_sales/store_sales.dat", 23),
}
raw = {}
for name, (path, expected_cols) in sources.items():
    df = (spark.read.option("header", "false").option("sep", "|")
          .option("inferSchema", "false").option("mode", "FAILFAST").csv(path))
    assert len(df.columns) == expected_cols, f"Schéma inattendu : {name}"
    raw[name] = df
    print(f"{name}: {df.count():,} lignes ; {len(df.columns)} colonnes")
raw["store"].select("_c0", "_c1", "_c2", "_c3").show(5, truncate=False)

store: 12 lignes ; 29 colonnes


item: 17,820 lignes ; 22 colonnes


store_sales: 667,579 lignes ; 23 colonnes
+---+----------------+----------+----+
|_c0|_c1             |_c2       |_c3 |
+---+----------------+----------+----+
|0  |AAAAAAAAAAAAAAAA|2000-08-02|NULL|
|1  |AAAAAAAAAAAAAAAB|2000-07-03|NULL|
|2  |AAAAAAAAAAAAAAAC|2000-09-01|NULL|
|3  |AAAAAAAAAAAAAAAD|2000-06-02|NULL|
|4  |AAAAAAAAAAAAAAAE|2000-07-03|NULL|
+---+----------------+----------+----+
only showing top 5 rows


### 3. Audit exploratoire des valeurs manquantes et des doublons

Les statistiques ci-dessous sont calculées sur **l’intégralité des fichiers**, pas uniquement un échantillon de 1 000 lignes.

In [3]:
audit = {}
for name, df in raw.items():
    n = df.count()
    null_counts = df.agg(*[
        F.sum(F.when(F.col(col).isNull() | (F.trim(F.col(col)) == ""), 1).otherwise(0)).alias(col)
        for col in df.columns
    ]).first().asDict()
    doublons = n - df.distinct().count()
    audit[name] = {"lignes": n, "doublons_complets": doublons,
                   "nulls": {k: v for k,v in null_counts.items() if v}}
    print(name, audit[name])

store {'lignes': 12, 'doublons_complets': 0, 'nulls': {'_c3': 12}}
item {'lignes': 17820, 'doublons_complets': 0, 'nulls': {'_c3': 17820, '_c4': 4, '_c7': 8, '_c8': 6}}


[Stage 45:>                                                         (0 + 2) / 3]

store_sales {'lignes': 667579, 'doublons_complets': 0, 'nulls': {'_c4': 238, '_c5': 295, '_c6': 448, '_c8': 158}}


### 4. Préparer la couche Silver

Normalisation des chaînes (`trim`, chaînes vides → `NULL`). Pour `store`, les colonnes `_c0` et `_c2` sont converties respectivement en entier et en date. Les autres colonnes restent des chaînes tant qu’un schéma officiel n est pas confirmé.

In [4]:
silver = {}
for name, df in raw.items():
    if name == "store":
        transformed = (df.withColumn("_c0", F.col("_c0").cast("int"))
                       .withColumn("_c2", F.to_date("_c2", "yyyy-MM-dd")))
        assert transformed.filter(F.col("_c0").isNull() | F.col("_c2").isNull()).count() == 0
    else:
        transformed = df.select(*[
            F.when(F.trim(F.col(x)) == "", F.lit(None))
             .otherwise(F.trim(F.col(x))).alias(x) for x in df.columns
        ])
    assert transformed.count() == audit[name]["lignes"]
    silver[name] = transformed
    print(name, "Silver :", transformed.count(), "lignes")

store Silver : 12 lignes
item Silver : 17820 lignes
store_sales Silver : 667579 lignes


### 5. Sauvegarde ou vérification Silver en Parquet


In [5]:
def hdfs_exists(path):
    jvm = spark._jvm
    conf = spark._jsc.hadoopConfiguration()
    p = jvm.org.apache.hadoop.fs.Path(path)
    return p.getFileSystem(conf).exists(p)

for name, df in silver.items():
    target = f"{BASE}/silver/uc06_{name}"
    if not hdfs_exists(target):
        df.write.mode("errorifexists").option("compression", "snappy").parquet(target)
        print("Créé :", target)
    else:
        print("Déjà présent, conservé :", target)
    reread = spark.read.parquet(target)
    assert reread.count() == df.count()
    print("Lecture Parquet validée :", name, reread.count())

Déjà présent, conservé : hdfs://localhost:9000/retail-lake/silver/uc06_store


Lecture Parquet validée : store 12
Déjà présent, conservé : hdfs://localhost:9000/retail-lake/silver/uc06_item


Lecture Parquet validée : item 17820
Déjà présent, conservé : hdfs://localhost:9000/retail-lake/silver/uc06_store_sales


Lecture Parquet validée : store_sales 667579


### 6. Vérifier l’intégrité des jointures

Clés candidates vérifiées sur 667 579 lignes : `store_sales._c2 = item._c0` et `store_sales._c7 = store._c0`.

In [6]:
v = silver["store_sales"].select(
    F.col("_c2").alias("article_id"), F.col("_c7").alias("magasin_id"),
    "_c10", "_c13", "_c15", "_c20", "_c22")
a = silver["item"].select(F.col("_c0").alias("article_id"))
m = silver["store"].select(F.col("_c0").cast("string").alias("magasin_id"))
assert a.distinct().count() == a.count()
assert m.distinct().count() == m.count()
enrichies = v.join(a, "article_id", "inner").join(m, "magasin_id", "inner")
assert enrichies.count() == v.count()
print("Jointure conservée :", enrichies.count(), "/", v.count())

Jointure conservée : 667579 / 667579


### 7. Types des indicateurs et contrôle de la formule

La relation `_c10 × _c13 = _c15` a été constatée sur toutes les lignes. Les interprétations « quantité », « prix », « ventes », « net » et « bénéfice » doivent toujours être rapprochées du dictionnaire métier officiel.

In [7]:
detail = (enrichies.select(
    "article_id", "magasin_id",
    F.expr("try_cast(_c10 as bigint)").alias("quantite"),
    F.expr("try_cast(_c13 as decimal(18,2))").alias("prix_unitaire"),
    F.expr("try_cast(_c15 as decimal(18,2))").alias("montant_ventes"),
    F.expr("try_cast(_c20 as decimal(18,2))").alias("montant_net"),
    F.expr("try_cast(_c22 as decimal(18,2))").alias("benefice")
))
invalides = detail.filter(
    F.col("article_id").isNull() | F.col("magasin_id").isNull() |
    F.col("quantite").isNull() | (F.col("quantite") <= 0) |
    F.col("prix_unitaire").isNull() | F.col("montant_ventes").isNull() |
    F.col("montant_net").isNull() | F.col("benefice").isNull()
).count()
ecarts = detail.filter(
    F.abs(F.col("quantite")*F.col("prix_unitaire") - F.col("montant_ventes")) > F.lit(0.01)
).count()
assert invalides == 0 and ecarts == 0
print("Lignes invalides :", invalides, "| Écarts de montant :", ecarts)
detail.show(5, truncate=False)

Lignes invalides : 0 | Écarts de montant : 0
+----------+----------+--------+-------------+--------------+-----------+--------+
|article_id|magasin_id|quantite|prix_unitaire|montant_ventes|montant_net|benefice|
+----------+----------+--------+-------------+--------------+-----------+--------+
|10001     |10        |31      |23.08        |715.48        |715.48     |157.48  |
|10000     |10        |54      |83.34        |4500.36       |4500.36    |-856.98 |
|15556     |10        |39      |83.24        |3246.36       |2629.55    |-470.56 |
|4597      |4         |20      |133.40       |2668.00       |2668.00    |1180.40 |
|3258      |4         |76      |11.85        |900.60        |900.60     |153.52  |
+----------+----------+--------+-------------+--------------+-----------+--------+
only showing top 5 rows


### 8. Agrégations Gold : comptages et montants

Deux granularités sont produites : par magasin et par article. Les agrégats sont contrôlés avant sauvegarde.

In [8]:
def calculer_gold(key):
    counts = enrichies.groupBy(key).agg(F.count("*").alias("nombre_lignes_ventes"))
    finance = detail.groupBy(key).agg(
        F.count("*").alias("nombre_lignes"),
        F.sum("quantite").alias("quantite_totale"),
        F.sum("montant_ventes").alias("total_ventes"),
        F.sum("montant_net").alias("total_net"),
        F.sum("benefice").alias("total_benefice"),
    )
    return counts, finance

gold_m_counts, gold_m_finance = calculer_gold("magasin_id")
gold_a_counts, gold_a_finance = calculer_gold("article_id")

total = v.count()
for df in (gold_m_counts, gold_a_counts):
    assert df.agg(F.sum("nombre_lignes_ventes")).first()[0] == total
for df in (gold_m_finance, gold_a_finance):
    assert df.agg(F.sum("nombre_lignes")).first()[0] == total

gold_m_finance.orderBy(F.desc("total_ventes")).show(12, truncate=False)

+----------+-------------+---------------+------------+-----------+--------------+
|magasin_id|nombre_lignes|quantite_totale|total_ventes|total_net  |total_benefice|
+----------+-------------+---------------+------------+-----------+--------------+
|7         |57589        |2915053        |109714870.06|98950137.76|-47643702.48  |
|9         |56040        |2830684        |107294834.63|96748356.68|-46015802.68  |
|4         |56211        |2832572        |107292630.20|96770105.59|-46245313.52  |
|8         |55737        |2821869        |107232486.60|96578622.89|-45837269.98  |
|0         |56018        |2830581        |107232320.14|96727851.94|-46086598.19  |
|3         |55397        |2800824        |106544945.30|96063990.59|-45725385.12  |
|1         |55676        |2817524        |106406449.75|95929844.66|-46172561.01  |
|11        |55473        |2790631        |106007960.27|95607843.48|-45346473.98  |
|6         |54574        |2780253        |105525800.61|94751173.08|-45805948.18  |
|5  

### 9. Vérifier ou écrire les quatre sorties Gold

Même règle : écrire uniquement si le dossier manque, sinon vérifier les données existantes.

In [9]:
gold_outputs = {
    "uc06_ventes_magasins": (gold_m_counts, "nombre_lignes_ventes"),
    "uc06_ventes_articles": (gold_a_counts, "nombre_lignes_ventes"),
    "uc06_finance_magasins": (gold_m_finance, "nombre_lignes"),
    "uc06_finance_articles": (gold_a_finance, "nombre_lignes"),
}
for name, (df, count_col) in gold_outputs.items():
    path = f"{BASE}/gold/{name}"
    if not hdfs_exists(path):
        df.write.mode("errorifexists").option("compression", "snappy").parquet(path)
        print("Créé :", name)
    else:
        print("Déjà présent, conservé :", name)
    reread = spark.read.parquet(path)
    assert reread.agg(F.sum(count_col)).first()[0] == total
    print("Vérifié :", name, reread.count(), "groupes")

Déjà présent, conservé : uc06_ventes_magasins
Vérifié : uc06_ventes_magasins 12 groupes
Déjà présent, conservé : uc06_ventes_articles
Vérifié : uc06_ventes_articles 17820 groupes
Déjà présent, conservé : uc06_finance_magasins
Vérifié : uc06_finance_magasins 12 groupes
Déjà présent, conservé : uc06_finance_articles
Vérifié : uc06_finance_articles 17820 groupes


### 10. Synthèse des contrôles financiers

Les totaux doivent coïncider entre les agrégations par magasin et par article. Les montants affichés sont en **unités monétaires non confirmées**.

In [10]:
cols = ["nombre_lignes", "quantite_totale", "total_ventes", "total_net", "total_benefice"]
def sums(df):
    x = df.agg(*[F.sum(k).alias(k) for k in cols]).first()
    return {k: x[k] for k in cols}
mag = sums(spark.read.parquet(f"{BASE}/gold/uc06_finance_magasins"))
art = sums(spark.read.parquet(f"{BASE}/gold/uc06_finance_articles"))
assert mag == art
for key,value in mag.items(): print(key, ":", value)
print("Cohérence magasins/articles : OK")

nombre_lignes : 667579
quantite_totale : 33755474
total_ventes : 1277224004.61
total_net : 1150529586.37
total_benefice : -552553374.82
Cohérence magasins/articles : OK


### 11. Comparaison des tailles Raw et Silver

Taille logique HDFS des fichiers, en octets. Une réduction négative signifie que Silver est plus volumineux que Raw.

In [11]:
def hdfs_size(path):
    jvm = spark._jvm
    p = jvm.org.apache.hadoop.fs.Path(path)
    fs = p.getFileSystem(spark._jsc.hadoopConfiguration())
    return fs.getContentSummary(p).getLength()

raw_paths = {name:path for name,(path,_) in sources.items()}
total_raw = total_silver = 0
for name, source in raw_paths.items():
    r = hdfs_size(source)
    s = hdfs_size(f"{BASE}/silver/uc06_{name}")
    total_raw += r; total_silver += s
    print(f"{name}: Raw {r:,} o / Silver {s:,} o / réduction {100*(r-s)/r:.2f} %")
print(f"Global : {100*(total_raw-total_silver)/total_raw:.2f} %")

store: Raw 3,386 o / Silver 10,451 o / réduction -208.65 %
item: Raw 6,799,286 o / Silver 3,641,846 o / réduction 46.44 %
store_sales: Raw 92,764,700 o / Silver 45,532,130 o / réduction 50.92 %
Global : 50.60 %


### 12. Synthèse et limites

**Résultats précédemment observés** : 12 magasins, 17 820 articles, 667 579 lignes de ventes, 33 755 474 unités calculées, 1 277 224 004,61 en montant des ventes et 1 150 529 586,37 en montant net. Les deux agrégations financières ont des totaux identiques.

**Limites :**
- Le dictionnaire officiel des 23 tables n’a pas encore été intégré : les noms métier des champs restent à confirmer.
- Les champs `_c3` de `store` et `item` sont vides dans les données analysées ; aucune imputation n’a été appliquée.
- Les bénéfices agrégés sont négatifs ; leur signification métier requiert une vérification.
- La comparaison Raw/Parquet mesure la taille, **pas la performance**.


In [12]:
spark.stop()
print("Session Spark arrêtée.")

Session Spark arrêtée.
